# Similarity Search and Distance Metrics

After embedding, text retrieval is **geometry**: each chunk is a point in $\mathbb{R}^d$; a user query is another point; you find the **nearest** points under a chosen metric. That operation — **k-nearest neighbor (k-NN) search** — is the core of semantic search and the retrieval step in RAG.

This notebook develops the math and intuition behind **cosine similarity**, **dot product**, and **Euclidean distance**; shows how **normalization** changes rankings; implements **brute-force** search in NumPy; compares metrics on real OpenAI embeddings; and introduces **approximate nearest neighbor (ANN)** indexes used at scale.

Prerequisites: **01. Introduction to Embeddings** in this section.

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)

---

## 1. The Retrieval Problem

Given:

- A **corpus** of $N$ vectors $\{\mathbf{v}_1, \ldots, \mathbf{v}_N\}$ (embedded document chunks)
- A **query** vector $\mathbf{q}$ (embedded user question)

Find the **top-k** corpus vectors most similar to $\mathbf{q}$.

```
Corpus points in R^d          Query point q
        ·  ·                        ★
      ·      ·
         ·  ·   →  return 3 closest dots to ★
```

"Similar" must be defined by a **metric** or **similarity function**. The choice affects which documents rank highest — and must align with how your embedding model was trained and evaluated.

---

## 2. Dot Product and Vector Norms

For vectors $\mathbf{a}, \mathbf{b} \in \mathbb{R}^d$:

**Dot product (inner product):**

$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{d} a_i b_i$$

**Euclidean norm (L2):**

$$\|\mathbf{a}\| = \sqrt{\sum_{i=1}^{d} a_i^2}$$

Geometrically, the dot product relates to the **angle** between vectors and their **magnitudes**. Longer vectors can have larger dot products even when pointing in different directions — that matters for retrieval.

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([2.0, 4.0, 6.0])   # same direction as a, twice the length
c = np.array([3.0, -1.0, 0.0])  # different direction

print("Dot products:")
print(f"  a · b = {np.dot(a, b):.1f}  (same direction, b is longer)")
print(f"  a · c = {np.dot(a, c):.1f}")
print(f"\nNorms: ||a||={np.linalg.norm(a):.3f}  ||b||={np.linalg.norm(b):.3f}  ||c||={np.linalg.norm(c):.3f}")

$\mathbf{b}$ points the same way as $\mathbf{a}$ but $\mathbf{a} \cdot \mathbf{b}$ is large partly because $\|\mathbf{b}\|$ is large. Raw dot product confounds **direction** and **length**.

---

## 3. Cosine Similarity

**Cosine similarity** measures the **angle** between vectors — orientation without magnitude:

$$\text{cosine}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$$

Range: **$-1$ to $1$**. For typical text embeddings (non-opposing meanings), scores often fall in **$0$ to $1$**.

| Value | Interpretation |
|-------|----------------|
| **1** | Same direction (max similarity) |
| **0** | Orthogonal (unrelated in this geometry) |
| **-1** | Opposite direction (rare for similar text) |

### 3.1 Why Cosine Is Default for Text

- Embedding models are often trained so **direction** encodes semantics
- Document length should not dominate ranking (longer ≠ automatically more similar)
- Vector DBs commonly default to `cosine` distance for text indexes

### 3.2 Cosine vs Angle

$$\cos(\theta) = \text{cosine}(\mathbf{a}, \mathbf{b})$$

Similarity depends only on $\theta$, not on $\|\mathbf{a}\|$ or $\|\mathbf{b}\|$.

In [ ]:
def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print(f"cosine(a, b) same direction: {cosine(a, b):.4f}")
print(f"cosine(a, c) different:     {cosine(a, c):.4f}")
print(f"\nScaling b by 10 does not change cosine:")
print(f"cosine(a, 10*b) = {cosine(a, 10*b):.4f}")

---

## 4. Euclidean (L2) Distance

**L2 distance** measures straight-line separation:

$$\text{L2}(\mathbf{a}, \mathbf{b}) = \|\mathbf{a} - \mathbf{b}\| = \sqrt{\sum_{i=1}^{d} (a_i - b_i)^2}$$

| Property | Note |
|----------|------|
| **Smaller** distance → more similar | Opposite of similarity scores |
| Sensitive to magnitude | Long vectors farther from origin |
| Common in vision / some indexes | Text often prefers cosine |

Vector DBs may report **cosine distance** $= 1 - \text{cosine}$ or **squared L2** — always check docs when comparing numeric scores across tools.

In [ ]:
def l2_distance(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(a - b))


print(f"L2(a, b) same direction: {l2_distance(a, b):.4f}")
print(f"L2(a, c) different:     {l2_distance(a, c):.4f}")
print(f"\nL2 is NOT invariant to scaling b:")
print(f"L2(a, 10*b) = {l2_distance(a, 10*b):.4f}")

---

## 5. Normalization and Inner Product Search

A **unit vector** has $\|\mathbf{a}\| = 1$. **L2-normalize** by dividing by the norm:

$$\hat{\mathbf{a}} = \frac{\mathbf{a}}{\|\mathbf{a}\|}$$

For unit vectors:

$$\text{cosine}(\hat{\mathbf{a}}, \hat{\mathbf{b}}) = \hat{\mathbf{a}} \cdot \hat{\mathbf{b}}$$

So **cosine similarity equals dot product** on normalized vectors. Many systems normalize once at index time and use fast **inner product (IP)** search.

| Metric | After L2-normalize |
|--------|-------------------|
| Cosine similarity | = dot product |
| L2 distance | Related monotonically for unit vectors |

In [ ]:
def normalize(v: np.ndarray) -> np.ndarray:
    return v / np.linalg.norm(v)


a_hat, b_hat, c_hat = normalize(a), normalize(b), normalize(c)

print("On unit vectors:")
print(f"  dot(a_hat, b_hat)     = {np.dot(a_hat, b_hat):.4f}")
print(f"  cosine(a, b)          = {cosine(a, b):.4f}")
print(f"  dot(a_hat, c_hat)     = {np.dot(a_hat, c_hat):.4f}")
print(f"  cosine(a, c)          = {cosine(a, c):.4f}")

---

## 6. Comparing Metrics — Ranking Can Change

When vectors are **not** normalized and lengths differ, dot product and L2 can rank candidates differently from cosine.

In [ ]:
query = np.array([1.0, 0.0])
candidates = {
    "short_same_dir": np.array([2.0, 0.0]),
    "long_same_dir":  np.array([100.0, 0.0]),
    "short_angled":   np.array([1.0, 1.0]),
}

print(f"{'candidate':18s} {'cosine':>8s} {'dot':>10s} {'L2 dist':>10s}")
for name, vec in candidates.items():
    print(
        f"{name:18s} {cosine(query, vec):8.4f} {np.dot(query, vec):10.2f} {l2_distance(query, vec):10.4f}"
    )

print("\nDot product favors long_same_dir (magnitude). Cosine treats both same-direction equally.")

---

## 7. Metric Choice in Vector Databases

| DB setting | Optimizes | Typical text use |
|------------|-----------|------------------|
| `cosine` | Angle / cosine similarity | **Default** |
| `ip` / inner product | Dot product | After normalizing embeddings |
| `l2` | Euclidean distance | Some models / vision |

**Rules:**

1. Use the metric recommended for your **embedding model**
2. Keep **index and query** on the same model and metric
3. Re-index if you change metric or normalization strategy

---

## 8. Brute-Force k-NN Search

Algorithm:

1. Embed query → $\mathbf{q}$
2. For each corpus vector $\mathbf{v}_i$, compute $\text{score}(\mathbf{q}, \mathbf{v}_i)$
3. Sort by score descending (cosine/dot) or distance ascending (L2)
4. Return top-k

**Complexity per query:** $O(N \cdot d)$ for $N$ vectors of dimension $d$.

| Corpus size | Brute force |
|-------------|-------------|
| < 10k chunks | Usually fine |
| 100k+ | Consider ANN index |
| Millions | ANN + sharding required |

In [ ]:
# Toy 2D corpus — no API needed
CORPUS_2D = {
    "auth": np.array([0.9, 0.1]),
    "db":   np.array([0.1, 0.9]),
    "api":  np.array([0.85, 0.2]),
    "ml":   np.array([0.2, 0.8]),
}

q2d = np.array([0.95, 0.05])  # query near "auth" region


def brute_force_knn(query: np.ndarray, corpus: dict[str, np.ndarray], k: int = 2):
    scores = [(label, cosine(query, vec)) for label, vec in corpus.items()]
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:k]


for label, score in brute_force_knn(q2d, CORPUS_2D, k=3):
    print(f"{score:.4f}  {label}")

---

## 9. Approximate Nearest Neighbor (ANN)

Exact k-NN compares $\mathbf{q}$ to **every** vector. **ANN** algorithms sacrifice perfect recall for speed.

| Algorithm family | Idea |
|------------------|------|
| **HNSW** | Graph navigation — jump toward nearest neighbors |
| **IVF** | Partition space into clusters; search only nearby clusters |
| **Product quantization** | Compress vectors to reduce memory |

```
Exact:   check all N vectors     → 100% recall, slow at scale
ANN:     check << N candidates   → 95–99% recall typical, much faster
```

Chroma, Pinecone, Qdrant, and pgvector use ANN internally above certain scales. Tune **ef**, **nprobe**, or vendor equivalents when recall drops (notebook 09).

---

## 10. Vectorized Search in NumPy

Replace Python loops with matrix operations — important for medium corpora in notebooks and batch evals.

In [ ]:
corpus_matrix = np.stack(list(CORPUS_2D.values()))  # shape (N, d)
labels = list(CORPUS_2D.keys())

q_norm = np.linalg.norm(q2d)
corpus_norms = np.linalg.norm(corpus_matrix, axis=1)
cosine_scores = (corpus_matrix @ q2d) / (corpus_norms * q_norm)

top_k = 2
top_idx = np.argsort(cosine_scores)[::-1][:top_k]

print("Vectorized top-2:")
for i in top_idx:
    print(f"  {cosine_scores[i]:.4f}  {labels[i]}")

Pattern for $N$ corpus vectors stacked as `corpus_matrix` of shape `(N, d)`:

$$\text{scores} = \frac{\mathbf{V} \mathbf{q}}{\|\mathbf{v}_i\| \|\mathbf{q}\|}$$

(element-wise division along rows)

---

## 11. Demonstration Setup (OpenAI Embeddings)

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"

CORPUS = [
    "Alembic manages database schema migrations for SQLAlchemy.",
    "JWT tokens enable stateless API authentication.",
    "FastAPI is a modern Python web framework for building APIs.",
    "Our Notes API stores notes in memory with REST endpoints.",
    "Pytest fixtures simplify database setup in integration tests.",
]


def embed_texts(texts: list[str]) -> np.ndarray:
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in resp.data])


def cosine_scores_matrix(corpus_vecs: np.ndarray, query_vec: np.ndarray) -> np.ndarray:
    qn = np.linalg.norm(query_vec)
    cn = np.linalg.norm(corpus_vecs, axis=1)
    return (corpus_vecs @ query_vec) / (cn * qn)


print("Ready.")

---

## 12. Demonstration: Semantic Search (Brute Force)

Real embeddings — query uses "version" and "tables"; top hit should be Alembic/migrations.

In [ ]:
doc_vectors = embed_texts(CORPUS)
query = "How do I version database tables?"
q_vec = embed_texts([query])[0]

scores = cosine_scores_matrix(doc_vectors, q_vec)
top_idx = np.argsort(scores)[::-1][:3]

print(f"Query: {query}\n")
for i in top_idx:
    print(f"{scores[i]:.4f}  {CORPUS[i]}")

---

## 13. Demonstration: Cosine vs L2 Ranking

On the same vectors, compare whether top-3 **order** differs between metrics.

In [ ]:
def rank_by_l2(corpus_vecs: np.ndarray, query_vec: np.ndarray, k: int = 3):
    dists = np.linalg.norm(corpus_vecs - query_vec, axis=1)
    idx = np.argsort(dists)[:k]
    return [(i, float(dists[i])) for i in idx]


def rank_by_cosine(corpus_vecs: np.ndarray, query_vec: np.ndarray, k: int = 3):
    s = cosine_scores_matrix(corpus_vecs, query_vec)
    idx = np.argsort(s)[::-1][:k]
    return [(i, float(s[i])) for i in idx]


print("Top-3 by COSINE (higher better):")
for i, sc in rank_by_cosine(doc_vectors, q_vec):
    print(f"  {sc:.4f}  {CORPUS[i][:50]}...")

print("\nTop-3 by L2 distance (lower better):")
for i, dist in rank_by_l2(doc_vectors, q_vec):
    print(f"  {dist:.4f}  {CORPUS[i][:50]}...")

On OpenAI text embeddings, rankings are often **similar** but not identical. Standardize on **cosine** unless your stack docs say otherwise.

---

## 14. Demonstration: Score Gaps and Top-k

Inspect gaps between 1st and 2nd place — small gaps mean ambiguous retrieval.

In [ ]:
queries = [
    "How do I version database tables?",
    "authentication tokens for APIs",
    "write integration tests",
]

for query in queries:
    qv = embed_texts([query])[0]
    s = cosine_scores_matrix(doc_vectors, qv)
    top2 = np.argsort(s)[::-1][:2]
    gap = s[top2[0]] - s[top2[1]]
    print(f"Q: {query}")
    print(f"   #1 {s[top2[0]]:.4f}  {CORPUS[top2[0]][:45]}...")
    print(f"   #2 {s[top2[1]]:.4f}  {CORPUS[top2[1]][:45]}...  gap={gap:.4f}\n")

---

## 15. Similarity Thresholds

Some pipelines return "no match" if the best score is below a threshold — avoids feeding irrelevant chunks to the LLM.

```python
THRESHOLD = 0.35  # calibrate on YOUR data — not universal
if best_score < THRESHOLD:
    return "No relevant documents found."
```

Thresholds depend on model, domain, and chunk size. Use notebook **09** eval sets to calibrate.

In [ ]:
THRESHOLD = 0.30  # demo only — tune in production

off_topic = "best pizza toppings in Italy"
qv = embed_texts([off_topic])[0]
s = cosine_scores_matrix(doc_vectors, qv)
best_i, best_score = int(np.argmax(s)), float(np.max(s))

print(f"Off-topic query: {off_topic}")
print(f"Best match score: {best_score:.4f}")
print(f"Best match text:  {CORPUS[best_i]}")
print(f"Pass threshold {THRESHOLD}? {best_score >= THRESHOLD}")

---

## 16. Score Interpretation Guide

| Observation | Likely cause | Action |
|-------------|--------------|--------|
| All scores low | Wrong model, bad chunks, off-domain query | Check ingest + eval set |
| Top scores very close | Ambiguous query | Rerank, hybrid search, ask clarifying question |
| Right doc ranks 2nd | Chunking or metric mismatch | Tune chunks; verify cosine |
| High score but wrong content | Similar topic, wrong answer | Metadata filters; better chunks |

Absolute cosine values are **not portable** across models — compare relative ranks on labeled queries.

---

## 17. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Raw dot product on unnormalized vectors | Length bias | Use cosine or normalize |
| Mixed metrics index vs query | Wrong rankings | One metric end-to-end |
| Ignoring k too small | Missed relevant chunk | Increase k; eval recall@k |
| Ignoring k too large | Noise in LLM context | Rerank; lower k |
| Universal threshold from blog | False rejects/accepts | Calibrate locally |

---

## 18. Summary

Retrieval is **k-nearest neighbor search** in embedding space. **Cosine similarity** measures angle and is the default for text. **Dot product** matches cosine on **L2-normalized** vectors. **L2 distance** measures separation and can rank differently when magnitudes vary.

**Brute-force** search is $O(Nd)$ per query — fine for small indexes; **ANN** indexes scale to millions. Vectorize with matrix multiply in NumPy; production uses Chroma and other DBs (notebooks 05–07).

Demonstrations covered toy 2D search, real OpenAI semantic search, cosine vs L2 ranking, score gaps, and thresholding.

Next: **03. Embedding APIs and Model Selection** — batching, dimensions, providers, and cost.